# Generate 2026 Fantasy Football Projections

This notebook builds 2026 player feature rows using only information
available before the 2026 regular season, then applies the selected
position-specific models to generate full-PPR season projections.

In [1]:
import polars as pl
import pandas as pd

df = pl.read_csv(
    "../data/processed/modeling_dataset_2018_2025.csv"
)

print(df.shape)

(2909, 83)


In [2]:
players_2025 = df.filter(
    pl.col("season") == 2025
)

print(players_2025.shape)
players_2025.group_by("position").len().sort("position")

(398, 83)


position,len
str,u32
"""QB""",58
"""RB""",95
"""TE""",94
"""WR""",151


In [3]:
projection_2026 = players_2025.with_columns(
    pl.lit(2026).alias("season")
)

In [4]:
players_2025_source = df.filter(
    pl.col("season") == 2025
)

players_2024_source = df.filter(
    pl.col("season") == 2024
)

In [5]:
source_2025 = players_2025_source.select([
    "player_id",

    pl.col("fantasy_points_ppr_calc").alias("fp_2025"),
    pl.col("games").alias("games_2025"),
    pl.col("targets").alias("targets_2025"),
    pl.col("carries").alias("carries_2025"),
    pl.col("receptions").alias("receptions_2025"),

    pl.col("target_share").alias("target_share_2025"),
    pl.col("air_yards_share").alias("air_yards_share_2025"),
    pl.col("wopr").alias("wopr_2025"),

    pl.col("yards_per_target").alias("yards_per_target_2025"),
    pl.col("yards_per_carry").alias("yards_per_carry_2025"),
    pl.col("catch_rate").alias("catch_rate_2025"),
    pl.col("fantasy_points_per_opportunity")
      .alias("fantasy_points_per_opportunity_2025"),

    pl.col("attempts").alias("attempts_2025"),
    pl.col("passing_yards").alias("passing_yards_2025"),
    pl.col("passing_tds").alias("passing_tds_2025"),
    pl.col("passing_interceptions").alias("passing_interceptions_2025")
])

In [6]:
source_2024 = players_2024_source.select([
    "player_id",

    pl.col("fantasy_points_ppr_calc").alias("fp_2024"),
    pl.col("games").alias("games_2024"),
    pl.col("targets").alias("targets_2024"),
    pl.col("carries").alias("carries_2024"),

    pl.col("target_share").alias("target_share_2024"),
    pl.col("wopr").alias("wopr_2024"),

    pl.col("attempts").alias("attempts_2024"),
    pl.col("passing_yards").alias("passing_yards_2024")
])

In [7]:
projection_2026 = (
    projection_2026
    .join(
        source_2025,
        on="player_id",
        how="left"
    )
    .join(
        source_2024,
        on="player_id",
        how="left"
    )
)

In [8]:
print(projection_2026.shape)

projection_2026.select([
    "player_display_name",
    "position",
    "season",
    "fp_2025",
    "fp_2024",
    "games_2025",
    "games_2024"
]).head(20)

(398, 107)


player_display_name,position,season,fp_2025,fp_2024,games_2025,games_2024
str,str,i32,f64,f64,i64,i64
"""Philip Rivers""","""QB""",2026,31.66,null,3,null
"""Aaron Rodgers""","""QB""",2026,227.08,256.58,16,17
"""Marcedes Lewis""","""TE""",2026,0.0,1.2,1,5
"""Joe Flacco""","""QB""",2026,146.66,99.04,13,7
"""Josh Johnson""","""QB""",2026,24.38,0.78,3,4
…,…,…,…,…,…,…
"""Geno Smith""","""QB""",2026,173.9,266.0,15,17
"""Brandin Cooks""","""WR""",2026,51.9,69.6,13,10
"""Teddy Bridgewater""","""QB""",2026,2.88,null,3,null


In [ ]:
# Core production history features

projection_2026 = projection_2026.with_columns([
    pl.col("fp_2025").alias("fantasy_points_lag_1"),
    pl.col("fp_2024").alias("fantasy_points_lag_2"),

    (
        pl.col("fp_2025") / pl.col("games_2025")
    ).alias("fantasy_ppg_lag_1"),

    (
        0.7 * pl.col("fp_2025")
        + 0.3 * pl.col("fp_2024")
    ).alias("fantasy_points_2yr_weighted"),

    pl.col("games_2025").alias("games_lag_1"),

    (
        (
            pl.col("games_2025")
            + pl.col("games_2024")
        ) / 2
    ).alias("games_2yr_avg"),

    (
        pl.col("fp_2025")
        - pl.col("fp_2024")
    ).alias("fantasy_points_change")
])

In [ ]:
# opportunity features

projection_2026 = projection_2026.with_columns([
    pl.col("targets_2025").alias("targets_lag_1"),
    pl.col("carries_2025").alias("carries_lag_1"),
    pl.col("receptions_2025").alias("receptions_lag_1"),

    (
        pl.col("targets_2025")
        + pl.col("carries_2025")
    ).alias("opportunities_lag_1"),

    (
        pl.col("targets_2025")
        / pl.col("games_2025")
    ).alias("targets_per_game_lag_1"),

    (
        pl.col("carries_2025")
        / pl.col("games_2025")
    ).alias("carries_per_game_lag_1"),

    (
        (
            pl.col("targets_2025")
            + pl.col("carries_2025")
        )
        / pl.col("games_2025")
    ).alias("opportunities_per_game_lag_1")
])

In [11]:
# two year opportunity averages

projection_2026 = projection_2026.with_columns([
    (
        (
            pl.col("targets_2025")
            + pl.col("targets_2024")
        ) / 2
    ).alias("targets_2yr_avg"),

    (
        (
            pl.col("carries_2025")
            + pl.col("carries_2024")
        ) / 2
    ).alias("carries_2yr_avg")
])

In [12]:
projection_2026 = projection_2026.with_columns(
    (
        pl.col("targets_2yr_avg")
        + pl.col("carries_2yr_avg")
    ).alias("opportunities_2yr_avg")
)

In [13]:
projection_2026 = projection_2026.with_columns([
    pl.col("target_share_2025").alias("target_share_lag_1"),
    pl.col("air_yards_share_2025").alias("air_yards_share_lag_1"),
    pl.col("wopr_2025").alias("wopr_lag_1"),

    (
        pl.col("target_share_2025")
        - pl.col("target_share_2024")
    ).alias("targets_change"),

    (
        (
            pl.col("target_share_2025")
            + pl.col("target_share_2024")
        ) / 2
    ).alias("target_share_2yr_avg"),

    (
        (
            pl.col("wopr_2025")
            + pl.col("wopr_2024")
        ) / 2
    ).alias("wopr_2yr_avg")
])

In [14]:
projection_2026 = projection_2026.with_columns(
    (
        pl.col("targets_2025")
        - pl.col("targets_2024")
    ).alias("targets_change")
)

In [15]:
projection_2026 = projection_2026.with_columns([
    pl.col("yards_per_target_2025").alias("yards_per_target_lag_1"),
    pl.col("yards_per_carry_2025").alias("yards_per_carry_lag_1"),
    pl.col("catch_rate_2025").alias("catch_rate_lag_1"),
    pl.col("fantasy_points_per_opportunity_2025")
      .alias("fantasy_points_per_opportunity_lag_1")
])

In [16]:
projection_2026.select([
    "player_display_name",
    "position",
    "fantasy_points_lag_1",
    "fantasy_points_lag_2",
    "fantasy_points_2yr_weighted",
    "games_lag_1",
    "targets_lag_1",
    "carries_lag_1",
    "targets_2yr_avg",
    "carries_2yr_avg"
]).head(20)

player_display_name,position,fantasy_points_lag_1,fantasy_points_lag_2,fantasy_points_2yr_weighted,games_lag_1,targets_lag_1,carries_lag_1,targets_2yr_avg,carries_2yr_avg
str,str,f64,f64,f64,i64,i64,i64,f64,f64
"""Philip Rivers""","""QB""",31.66,null,null,3,0,2,null,null
"""Aaron Rodgers""","""QB""",227.08,256.58,235.93,16,1,21,0.5,21.5
"""Marcedes Lewis""","""TE""",0.0,1.2,0.36,1,0,0,0.5,0.0
"""Joe Flacco""","""QB""",146.66,99.04,132.374,13,0,21,0.0,15.0
"""Josh Johnson""","""QB""",24.38,0.78,17.3,3,0,12,0.0,8.0
…,…,…,…,…,…,…,…,…,…
"""Geno Smith""","""QB""",173.9,266.0,201.53,15,0,41,0.0,47.0
"""Brandin Cooks""","""WR""",51.9,69.6,57.21,13,36,0,45.0,1.5
"""Teddy Bridgewater""","""QB""",2.88,null,null,3,0,3,null,null


# Loading just the 2026 Roster

In [17]:
import nflreadpy as nfl

rosters_2026 = nfl.load_rosters(seasons=[2026])

print(rosters_2026.shape)

(2930, 36)


In [18]:
roster_2026 = (
    rosters_2026
    .select([
        "gsis_id",
        "full_name",
        "team",
        "birth_date",
        "years_exp",
        "draft_number"
    ])
    .unique(
        subset=["gsis_id"],
        keep="first"
    )
)

In [19]:
roster_2026 = roster_2026.rename({
    "gsis_id": "player_id",
    "team": "team_2026",
    "birth_date": "birth_date_2026",
    "years_exp": "years_exp_2026",
    "draft_number": "draft_number_2026"
})

In [21]:
# inner join so it removes players who appeared in 2025 but are not on a 2026 roster

projection_2026 = projection_2026.join(
    roster_2026,
    on="player_id",
    how="inner"
)

In [22]:
print(projection_2026.shape)

projection_2026.group_by("position").len().sort("position")

(325, 117)


position,len
str,u32
"""QB""",51
"""RB""",73
"""TE""",82
"""WR""",119


In [24]:
projection_2026 = projection_2026.with_columns(
    pl.col("birth_date_2026")
      .alias("birth_date_2026")
)

In [25]:
projection_2026 = projection_2026.with_columns(
    (
        (
            pl.date(2026, 9, 1)
            - pl.col("birth_date_2026")
        )
        .dt.total_days()
        / 365.25
    ).alias("age")
)

In [26]:
projection_2026 = projection_2026.with_columns(
    (pl.col("age") ** 2).alias("age_squared")
)

In [27]:
projection_2026 = projection_2026.with_columns(
    pl.col("years_exp_2026").alias("years_exp")
)

In [28]:
projection_2026 = projection_2026.with_columns(
    (
        pl.col("team_2026") != pl.col("recent_team")
    )
    .cast(pl.Int8)
    .alias("team_change_flag")
)

In [29]:
projection_2026 = projection_2026.with_columns([
    pl.col("draft_number_2026")
      .is_null()
      .cast(pl.Int8)
      .alias("undrafted_flag"),

    pl.col("draft_number_2026")
      .fill_null(260)
      .alias("draft_number_filled")
])

In [30]:
projection_2026.select([
    "player_display_name",
    "position",
    "recent_team",
    "team_2026",
    "team_change_flag",
    "age",
    "years_exp",
    "draft_number_filled",
    "undrafted_flag"
]).head(20)

player_display_name,position,recent_team,team_2026,team_change_flag,age,years_exp,draft_number_filled,undrafted_flag
str,str,str,str,i8,f64,i32,i32,i8
"""Mack Hollins""","""WR""","""NE""","""NE""",0,32.958248,9,118,0
"""Hunter Long""","""TE""","""JAX""","""JAX""",0,28.035592,5,81,0
"""Olamide Zaccheaus""","""WR""","""CHI""","""ATL""",1,29.10883,7,260,1
"""Michael Wilson""","""WR""","""ARI""","""AZ""",1,26.521561,3,94,0
"""Xavier Hutchinson""","""WR""","""HOU""","""HOU""",0,26.250513,3,205,0
…,…,…,…,…,…,…,…,…
"""Reggie Gilliam""","""RB""","""BUF""","""NE""",1,29.03217,6,260,1
"""Jordan Love""","""QB""","""GB""","""GB""",0,27.830253,6,26,0
"""Luke Farrell""","""TE""","""SF""","""SF""",0,28.881588,5,145,0


In [31]:
projection_2026.select([
    "age",
    "age_squared",
    "years_exp",
    "draft_number_filled",
    "undrafted_flag",
    "team_change_flag"
]).null_count()

age,age_squared,years_exp,draft_number_filled,undrafted_flag,team_change_flag
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


# Final Feature Availability QA

In [32]:
base_features = [
    "age",
    "age_squared",
    "years_exp",
    "draft_number_filled",
    "undrafted_flag",
    "team_change_flag",
    "fantasy_points_lag_1",
    "fantasy_points_lag_2",
    "fantasy_ppg_lag_1",
    "fantasy_points_2yr_weighted",
    "games_lag_1",
    "games_2yr_avg",
    "fantasy_points_change"
]

qb_features = base_features + [
    "carries_lag_1",
    "carries_per_game_lag_1",
    "carries_2yr_avg"
]

rb_features = base_features + [
    "targets_lag_1",
    "carries_lag_1",
    "receptions_lag_1",
    "opportunities_lag_1",
    "targets_per_game_lag_1",
    "carries_per_game_lag_1",
    "opportunities_per_game_lag_1",
    "yards_per_carry_lag_1",
    "fantasy_points_per_opportunity_lag_1",
    "targets_2yr_avg",
    "carries_2yr_avg",
    "opportunities_2yr_avg"
]

wr_features = base_features + [
    "targets_lag_1",
    "receptions_lag_1",
    "targets_per_game_lag_1",
    "target_share_lag_1",
    "air_yards_share_lag_1",
    "wopr_lag_1",
    "targets_change",
    "yards_per_target_lag_1",
    "catch_rate_lag_1",
    "fantasy_points_per_opportunity_lag_1",
    "targets_2yr_avg",
    "target_share_2yr_avg",
    "wopr_2yr_avg"
]

te_features = wr_features.copy()

In [33]:
position_feature_sets = {
    "QB": qb_features,
    "RB": rb_features,
    "WR": wr_features,
    "TE": te_features
}

for position, features in position_feature_sets.items():
    missing_columns = [
        feature
        for feature in features
        if feature not in projection_2026.columns
    ]

    print(position, "missing columns:", missing_columns)

QB missing columns: []
RB missing columns: []
WR missing columns: []
TE missing columns: []


In [34]:
for position, features in position_feature_sets.items():
    pos_df = projection_2026.filter(
        pl.col("position") == position
    )

    print(f"\n{position}")
    print(
        pos_df
        .select(features)
        .null_count()
    )


QB
shape: (1, 16)
┌─────┬────────────┬───────────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ age ┆ age_square ┆ years_exp ┆ draft_numb ┆ … ┆ fantasy_po ┆ carries_la ┆ carries_pe ┆ carries_2 │
│ --- ┆ d          ┆ ---       ┆ er_filled  ┆   ┆ ints_chang ┆ g_1        ┆ r_game_lag ┆ yr_avg    │
│ u32 ┆ ---        ┆ u32       ┆ ---        ┆   ┆ e          ┆ ---        ┆ _1         ┆ ---       │
│     ┆ u32        ┆           ┆ u32        ┆   ┆ ---        ┆ u32        ┆ ---        ┆ u32       │
│     ┆            ┆           ┆            ┆   ┆ u32        ┆            ┆ u32        ┆           │
╞═════╪════════════╪═══════════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 0   ┆ 0          ┆ 0         ┆ 0          ┆ … ┆ 8          ┆ 0          ┆ 0          ┆ 8         │
└─────┴────────────┴───────────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘

RB
shape: (1, 25)
┌─────┬────────────┬───────────┬────────────┬───┬────

# 2026 Model Training

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

In [36]:
historical_df = pl.read_csv(
    "../data/processed/modeling_dataset_2018_2025.csv"
)

model_data = historical_df.to_pandas()

In [37]:
qb_train = model_data[
    model_data["position"] == "QB"
].copy()

rb_train = model_data[
    model_data["position"] == "RB"
].copy()

wr_train = model_data[
    model_data["position"] == "WR"
].copy()

te_train = model_data[
    model_data["position"] == "TE"
].copy()

In [38]:
final_qb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])

final_rb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=30))
])

final_wr_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])

final_te_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

In [39]:
target = "fantasy_points_ppr_calc"

In [40]:
final_qb_model.fit(
    qb_train[qb_features],
    qb_train[target]
)

final_rb_model.fit(
    rb_train[rb_features],
    rb_train[target]
)

final_wr_model.fit(
    wr_train[wr_features],
    wr_train[target]
)

final_te_model.fit(
    te_train[te_features],
    te_train[target]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](26,)","['age','age_squared','years_exp',...,'targets_2yr_avg', 'target_share_2yr_avg','wopr_2yr_avg']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,26
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or 

In [41]:
projection_2026_pd = projection_2026.to_pandas()

In [42]:
qb_2026 = projection_2026_pd[
    projection_2026_pd["position"] == "QB"
].copy()

rb_2026 = projection_2026_pd[
    projection_2026_pd["position"] == "RB"
].copy()

wr_2026 = projection_2026_pd[
    projection_2026_pd["position"] == "WR"
].copy()

te_2026 = projection_2026_pd[
    projection_2026_pd["position"] == "TE"
].copy()

In [43]:
qb_2026["projected_fantasy_points"] = final_qb_model.predict(
    qb_2026[qb_features]
)

rb_2026["projected_fantasy_points"] = final_rb_model.predict(
    rb_2026[rb_features]
)

wr_2026["projected_fantasy_points"] = final_wr_model.predict(
    wr_2026[wr_features]
)

te_2026["projected_fantasy_points"] = final_te_model.predict(
    te_2026[te_features]
)

In [44]:
projections_2026 = pd.concat([
    qb_2026,
    rb_2026,
    wr_2026,
    te_2026
], ignore_index=True)

# 2026 PROJECTIONS

In [45]:
projection_view = projections_2026[
    [
        "player_display_name",
        "team_2026",
        "position",
        "projected_fantasy_points"
    ]
].sort_values(
    "projected_fantasy_points",
    ascending=False
)

projection_view.head(30)

,player_display_name,team_2026,position,projected_fantasy_points
19,Josh Allen,BUF,QB,317.416663
164,Ja'Marr Chase,CIN,WR,299.661306
145,Amon-Ra St. Brown,DET,WR,274.410290
166,Jaxon Smith-Njigba,SEA,WR,272.587505
15,Jared Goff,DET,QB,271.206485
31,Matthew Stafford,LA,QB,266.523483
35,Baker Mayfield,TB,QB,264.326834
38,Jalen Hurts,PHI,QB,263.150776
16,Lamar Jackson,BAL,QB,260.002042
109,Christian McCaffrey,SF,RB,259.788606


In [46]:
projections_2026["projected_fantasy_points"] = (
    projections_2026["projected_fantasy_points"]
    .round(1)
)

In [47]:
projections_2026["position_rank"] = (
    projections_2026
    .groupby("position")["projected_fantasy_points"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

In [48]:
projections_2026["position_rank_label"] = (
    projections_2026["position"]
    + projections_2026["position_rank"].astype(str)
)

In [49]:
for position in ["QB", "RB", "WR", "TE"]:

    display(
        projections_2026[
            projections_2026["position"] == position
        ][[
            "position_rank_label",
            "player_display_name",
            "team_2026",
            "projected_fantasy_points"
        ]]
        .sort_values(
            "projected_fantasy_points",
            ascending=False
        )
        .head(15)
    )

,position_rank_label,player_display_name,team_2026,projected_fantasy_points
19,QB1,Josh Allen,BUF,317.4
15,QB2,Jared Goff,DET,271.2
31,QB3,Matthew Stafford,LA,266.5
35,QB4,Baker Mayfield,TB,264.3
38,QB5,Jalen Hurts,PHI,263.2
16,QB6,Lamar Jackson,BAL,260.0
7,QB7,Patrick Mahomes,KC,255.4
8,QB8,Justin Herbert,LAC,251.1
46,QB9,Trevor Lawrence,JAX,242.4
5,QB10,Sam Darnold,SEA,235.7


,position_rank_label,player_display_name,team_2026,projected_fantasy_points
109,RB1,Christian McCaffrey,SF,259.8
97,RB2,Jonathan Taylor,IND,240.0
73,RB3,James Cook,BUF,215.7
123,RB4,Bijan Robinson,ATL,205.8
99,RB5,Saquon Barkley,PHI,196.5
60,RB6,Javonte Williams,DAL,195.6
63,RB7,Jahmyr Gibbs,DET,194.4
70,RB8,Derrick Henry,BAL,189.3
78,RB9,Josh Jacobs,GB,188.5
91,RB10,Breece Hall,NYJ,187.4


,position_rank_label,player_display_name,team_2026,projected_fantasy_points
164,WR1,Ja'Marr Chase,CIN,299.7
145,WR2,Amon-Ra St. Brown,DET,274.4
166,WR3,Jaxon Smith-Njigba,SEA,272.6
179,WR4,Puka Nacua,LA,242.1
133,WR5,Justin Jefferson,MIN,223.1
143,WR6,Drake London,ATL,217.9
181,WR7,George Pickens,DAL,204.5
232,WR8,Zay Flowers,BAL,203.8
214,WR9,Jameson Williams,DET,197.9
178,WR11,Nico Collins,HOU,197.7


,position_rank_label,player_display_name,team_2026,projected_fantasy_points
314,TE1,Trey McBride,AZ,216.8
290,TE2,Kyle Pitts,ATL,167.6
261,TE3,Travis Kelce,KC,154.8
244,TE4,Hunter Henry,NE,148.1
288,TE5,George Kittle,SF,142.1
267,TE6,Juwan Johnson,NO,137.9
302,TE7,Dallas Goedert,PHI,137.2
255,TE8,Jake Ferguson,DAL,135.8
310,TE9,Dalton Schultz,HOU,124.5
296,TE10,Cade Otton,TB,124.3


In [50]:
final_projections_2026 = projections_2026[
    [
        "player_id",
        "player_display_name",
        "team_2026",
        "position",
        "position_rank",
        "position_rank_label",
        "projected_fantasy_points"
    ]
].copy()

final_projections_2026 = final_projections_2026.sort_values(
    ["position", "position_rank"]
)

final_projections_2026.head()

,player_id,player_display_name,team_2026,position,position_rank,position_rank_label,projected_fantasy_points
19,00-0034857,Josh Allen,BUF,QB,1,QB1,317.4
15,00-0033106,Jared Goff,DET,QB,2,QB2,271.2
31,00-0026498,Matthew Stafford,LA,QB,3,QB3,266.5
35,00-0034855,Baker Mayfield,TB,QB,4,QB4,264.3
38,00-0036389,Jalen Hurts,PHI,QB,5,QB5,263.2


In [51]:
final_projections_2026.to_csv(
    "../data/processed/2026_fantasy_projections.csv",
    index=False
)

2026 Projection Output: Final position-specific models were retrained using all eligible historical data through the 2025 season. These models were applied to 2026 player rows constructed entirely from preseason-available information and historical 2024–2025 production. The resulting projections represent expected full-PPR regular-season fantasy points.